# MonoDETR R0 Vehicle + Pedestrian reference

This is the frozen ResNet50 accuracy-reference run for MobileADAS3D-S1. It fine-tunes the published MonoDETR checkpoint on the Chen 3,712-image train split after mapping Car/Van/Truck/Tram to the native Car ID and Pedestrian/Person_sitting to the native Pedestrian ID. Use a GPU runtime. Do not change parameters without assigning a new reference ID.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, shlex, shutil, subprocess, sys
MOBILE_REPO = Path('/content/mobile_adas3d')
MONODETR_REPO = Path('/content/MonoDETR')
MONODETR_COMMIT = '6994b9f512400b258c6edb75f77423beb9c126f2'
DRIVE_DATASET_ROOT = Path('/content/drive/MyDrive/datasets/kitti')
LOCAL_DATASET_ROOT = Path('/content/kitti')
SPLIT_DIR = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
MONODETR_KITTI = Path('/content/monodetr_kitti')
OFFICIAL_CHECKPOINT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/teachers/monodetr/checkpoint_best.pth')
OUTPUT_ROOT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/references/monodetr_r0')
RUN_NAME = 'monodetr_r0_vehicle_pedestrian'
MAX_EPOCHS = 195
SAVE_FREQUENCY = 5
BATCH_SIZE = 16
def run(command, cwd=None, env=None):
    command = [str(x) for x in command]; print('+', shlex.join(command), flush=True)
    merged = os.environ.copy(); merged.update(env or {})
    result = subprocess.run(command, cwd=cwd, env=merged)
    if result.returncode: raise RuntimeError(f'Exit {result.returncode}: {shlex.join(command)}')
run(['nvidia-smi'])

In [ ]:
# Fetch pinned source and apply current-Colab plus product-taxonomy patches.
if not MOBILE_REPO.exists(): run(['git', 'clone', 'https://github.com/Ali-RT/mobile_adas3d.git', MOBILE_REPO])
else: run(['git', 'pull', '--ff-only'], cwd=MOBILE_REPO)
if not MONODETR_REPO.exists(): run(['git', 'clone', 'https://github.com/ZrrSkywalker/MonoDETR.git', MONODETR_REPO])
run(['git', 'fetch', '--all'], cwd=MONODETR_REPO)
run(['git', 'checkout', MONODETR_COMMIT], cwd=MONODETR_REPO)
run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown', 'pyyaml', 'scipy', 'opencv-python-headless', 'numba', 'scikit-image', 'tqdm', 'ninja'])
run([sys.executable, 'scripts/patch_monodetr_colab_compat.py', '--monodetr-repo', MONODETR_REPO], cwd=MOBILE_REPO)
run([sys.executable, 'scripts/patch_monodetr_product_taxonomy.py', '--monodetr-repo', MONODETR_REPO], cwd=MOBILE_REPO)
run([sys.executable, 'scripts/patch_monodetr_verbose_resume.py', '--monodetr-repo', MONODETR_REPO], cwd=MOBILE_REPO)
run([sys.executable, 'scripts/patch_monodetr_checkpoint_metadata.py', '--monodetr-repo', MONODETR_REPO], cwd=MOBILE_REPO)
ops = MONODETR_REPO / 'lib/models/monodetr/ops'
shutil.rmtree(ops / 'build', ignore_errors=True)
run([sys.executable, 'setup.py', 'build', 'install'], cwd=ops, env={'MAX_JOBS': '2'})
run([sys.executable, '-c', 'import torch, MultiScaleDeformableAttention; from lib.models.monodetr import build_monodetr; print(torch.__version__, torch.cuda.get_device_name(0))'], cwd=MONODETR_REPO)

In [ ]:
# Create a zero-copy KITTI view from local staging when available, otherwise Drive.
def resolve(root, names):
    for name in names:
        path = root / name
        if path.is_dir(): return path
sources = {}
for key, names in {'image_2':['training/image_2','training/image_02'], 'label_2':['training/label_2','training/label_02'], 'calib':['training/calib']}.items():
    sources[key] = resolve(LOCAL_DATASET_ROOT, names) or resolve(DRIVE_DATASET_ROOT, names)
if any(path is None for path in sources.values()): raise FileNotFoundError(f'Missing KITTI sources: {sources}')
(MONODETR_KITTI/'training').mkdir(parents=True, exist_ok=True)
(MONODETR_KITTI/'ImageSets').mkdir(parents=True, exist_ok=True)
for name, target in sources.items():
    link = MONODETR_KITTI/'training'/name
    if link.is_symlink() and link.resolve() == target.resolve(): continue
    if link.exists() or link.is_symlink(): raise RuntimeError(f'Refusing to replace {link}')
    link.symlink_to(target, target_is_directory=True)
for split in ('train','val'): shutil.copy2(SPLIT_DIR/f'{split}.txt', MONODETR_KITTI/'ImageSets'/f'{split}.txt')
assert len((MONODETR_KITTI/'ImageSets/train.txt').read_text().splitlines()) == 3712
assert len((MONODETR_KITTI/'ImageSets/val.txt').read_text().splitlines()) == 3769
print('KITTI R0 view:', MONODETR_KITTI, sources)

In [ ]:
# Build the immutable R0 initialization, resolved config, and hash manifest.
if not OFFICIAL_CHECKPOINT.is_file(): raise FileNotFoundError(OFFICIAL_CHECKPOINT)
run([sys.executable, 'scripts/prepare_monodetr_r0_reference.py', '--monodetr-repo', MONODETR_REPO, '--dataset-root', MONODETR_KITTI, '--official-checkpoint', OFFICIAL_CHECKPOINT, '--output-root', OUTPUT_ROOT, '--run-name', RUN_NAME, '--max-epochs', MAX_EPOCHS, '--save-frequency', SAVE_FREQUENCY, '--batch-size', BATCH_SIZE], cwd=MOBILE_REPO)
CONFIG = MONODETR_REPO/'configs/monodetr_r0_vehicle_pedestrian.yaml'
RUN_DIR = OUTPUT_ROOT/RUN_NAME
print(CONFIG.read_text())
print((RUN_DIR/'experiment_manifest.json').read_text())

## Real R0 training

The next cell is the full GPU training loop. Expect per-batch losses, epoch progress, validation every 5 epochs, and durable Drive checkpoints. Batch size 16 is the frozen setting. If it does not fit, stop and report the GPU and error before changing it.

In [ ]:
# Automatically resume the newest complete Drive checkpoint. A failed or partial file is skipped.
import re, torch, yaml
checkpoint_pattern = re.compile(r'^checkpoint_epoch_(\d+)\.pth$')
valid_checkpoints = []
for path in RUN_DIR.glob('checkpoint_epoch_*.pth'):
    match = checkpoint_pattern.match(path.name)
    if not match: continue
    try:
        payload = torch.load(path, map_location='cpu', weights_only=False)
        epoch = int(payload.get('epoch', -1))
        if epoch != int(match.group(1)): raise ValueError(f'filename epoch {match.group(1)} != payload epoch {epoch}')
        if payload.get('model_state') is None: raise ValueError('model_state missing')
        if payload.get('optimizer_state') is None: raise ValueError('optimizer_state missing')
        valid_checkpoints.append((epoch, path))
        print(f'Valid resume checkpoint: epoch={epoch} size={path.stat().st_size/1e6:.1f}MB {path}')
    except Exception as error:
        print(f'Skipping invalid checkpoint {path}: {type(error).__name__}: {error}')
latest = max(valid_checkpoints, default=None, key=lambda item: item[0])
run_cfg = yaml.safe_load(CONFIG.read_text())
if latest is None:
    START_EPOCH = 0
    CONFIG_TO_RUN = CONFIG
    print('Starting fresh from the published MonoDETR R0 initialization.')
else:
    START_EPOCH, RESUME_CHECKPOINT = latest
    run_cfg['trainer'].pop('pretrain_model', None)
    run_cfg['trainer']['resume_model'] = str(RESUME_CHECKPOINT)
    run_cfg['trainer']['max_epoch'] = MAX_EPOCHS
    CONFIG_TO_RUN = MONODETR_REPO/'configs/monodetr_r0_vehicle_pedestrian_resume.yaml'
    CONFIG_TO_RUN.write_text(yaml.safe_dump(run_cfg, sort_keys=False))
    print(f'Resuming R0 after completed epoch {START_EPOCH}: {RESUME_CHECKPOINT}')
print('Training config:', CONFIG_TO_RUN)
print(f'Remaining epochs: {max(0, MAX_EPOCHS - START_EPOCH)}')

In [ ]:
# Real training with live output plus a durable combined stdout/stderr log.
from collections import deque
from datetime import datetime, timezone
LOG_DIR = OUTPUT_ROOT/'colab_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
def run_training_logged(command, cwd):
    command = [str(x) for x in command]
    stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
    log_path = LOG_DIR/f'train_{RUN_NAME}_{stamp}.log'
    print('+', shlex.join(command), flush=True)
    print('Durable combined log:', log_path, flush=True)
    env = os.environ.copy(); env['PYTHONUNBUFFERED'] = '1'
    tail = deque(maxlen=120)
    with log_path.open('w', encoding='utf-8', buffering=1) as log:
        process = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line); tail.append(line.rstrip())
        return_code = process.wait()
    if return_code:
        print('\n===== TRAINING FAILURE DIAGNOSTICS =====', flush=True)
        print('Return code:', return_code, '(negative means terminated by signal)')
        print('Command:', shlex.join(command))
        print('Working directory:', cwd)
        print('Config exists:', Path(CONFIG_TO_RUN).is_file(), CONFIG_TO_RUN)
        print('Log exists/bytes:', log_path.is_file(), log_path.stat().st_size if log_path.is_file() else 0)
        print('\nLast captured lines:')
        print('\n'.join(tail) if tail else '<child process emitted no output>')
        print('\nGPU state:')
        subprocess.run(['nvidia-smi'])
        print('\nDisk state:')
        subprocess.run(['df', '-h', '/content', '/content/drive'])
        print('\nRecent MonoDETR text logs:')
        for path in sorted(MONODETR_REPO.rglob('*.log'), key=lambda p: p.stat().st_mtime)[-10:]:
            print(path, path.stat().st_size, 'bytes')
        raise RuntimeError(f'Training exited {return_code}; inspect durable log: {log_path}')
    print('Training command completed. Durable log:', log_path)
    return log_path
# Re-run the previous resume-detection cell and this cell after interruption.
if START_EPOCH >= MAX_EPOCHS:
    print(f'R0 already reached epoch {START_EPOCH}; no training required.')
else:
    TRAIN_LOG = run_training_logged([sys.executable, '-u', 'tools/train_val.py', '--config', CONFIG_TO_RUN], cwd=MONODETR_REPO)

In [ ]:
# Durable artifacts. Product-taxonomy checkpoint sweep is the next gate.
print('Run directory:', RUN_DIR)
for path in sorted(RUN_DIR.glob('checkpoint*.pth')):
    print(path.name, round(path.stat().st_size / 1e6, 1), 'MB')
print('Do not select checkpoint_best.pth as final R0 yet: upstream selects Car AP only.')

## Product-taxonomy checkpoint sweep

Run after all 195 training epochs complete. This performs inference for every five-epoch checkpoint, maps native Car predictions to Vehicle, evaluates Vehicle/Pedestrian moderate 3D AP_R40, and selects the highest balanced mean. Completed epoch evaluations are cached, so the sweep is restartable. No retraining occurs.

In [ ]:
SWEEP_DIR = OUTPUT_ROOT/'product_checkpoint_sweep'
PRODUCT_CONFIG = 'configs/kitti_mobileadas3d_s1.yaml'
run([sys.executable, '-u', 'scripts/sweep_monodetr_r0_product_checkpoints.py', '--monodetr-repo', MONODETR_REPO, '--mobile-repo', MOBILE_REPO, '--training-config', CONFIG, '--run-dir', RUN_DIR, '--dataset-root', MONODETR_KITTI, '--split-dir', SPLIT_DIR, '--output-dir', SWEEP_DIR, '--product-config', PRODUCT_CONFIG, '--profile', 'colab_drive', '--score-threshold', '0.001', '--topk', '50'], cwd=MOBILE_REPO)
print((SWEEP_DIR/'r0_product_selection.json').read_text())

In [ ]:
# Compact ranking to share for review.
import pandas as pd
ranking = pd.read_csv(SWEEP_DIR/'r0_product_checkpoint_sweep.csv')
display(ranking[['rank','epoch','vehicle_3d_moderate','pedestrian_3d_moderate','mean_3d_moderate','vehicle_bev_moderate','pedestrian_bev_moderate']].head(15))
print('Selected checkpoint:', (SWEEP_DIR/'SELECTED_CHECKPOINT_PATH.txt').read_text().strip())

## Locked R0 epoch-185 qualification

Run cells 2-5 first, then this final cell. It performs no training and changes no model, threshold, TopK, split, taxonomy, or matching setting. It regenerates all 3,769 predictions and writes nearby-recall, geometry, yaw, and Pedestrian-failure artifacts to Drive.


In [ ]:
# Frozen R0 evaluation only: complete inference plus locked quality diagnostics.
import hashlib, json, pandas as pd, yaml
from collections import deque
from datetime import datetime, timezone

SELECTION_PATH = OUTPUT_ROOT/"product_checkpoint_sweep/r0_product_selection.json"
selection = json.loads(SELECTION_PATH.read_text())
selected = selection["selected"]
SELECTED_EPOCH = int(selected["epoch"])
SELECTED_CHECKPOINT = Path(selected["checkpoint"])
SELECTED_SHA256 = selected["checkpoint_sha256"]
if SELECTED_EPOCH != 185: raise RuntimeError(f"Expected frozen R0 epoch 185, found {SELECTED_EPOCH}")
if SELECTED_SHA256 != "fc0eba200e44b88921af76b0a5c94279872fd5c4838ab4d8936838447debfa59": raise RuntimeError(SELECTED_SHA256)
if not SELECTED_CHECKPOINT.is_file(): raise FileNotFoundError(SELECTED_CHECKPOINT)
def sha256(path):
    digest=hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024*1024), b""): digest.update(block)
    return digest.hexdigest()
if sha256(SELECTED_CHECKPOINT) != SELECTED_SHA256: raise RuntimeError("Frozen R0 checkpoint hash mismatch")

LOG_DIR=OUTPUT_ROOT/"qualification_logs"; LOG_DIR.mkdir(parents=True,exist_ok=True)
def run_logged(command,cwd):
    command=[str(x) for x in command]
    stamp=datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    log_path=LOG_DIR/f"r0_qualification_{stamp}.log"
    print("+",shlex.join(command),"\nDurable combined log:",log_path,flush=True)
    env=os.environ.copy(); env["PYTHONUNBUFFERED"]="1"; tail=deque(maxlen=120)
    with log_path.open("w",encoding="utf-8",buffering=1) as log:
        process=subprocess.Popen(command,cwd=cwd,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end="",flush=True); log.write(line); tail.append(line.rstrip())
        return_code=process.wait()
    if return_code:
        print("Last captured lines:\n"+"\n".join(tail))
        raise RuntimeError(f"R0 qualification inference exited {return_code}; log={log_path}")
    return log_path

diag_cfg=yaml.safe_load(CONFIG.read_text())
diag_cfg["tester"].update({"mode":"single","checkpoint":185,"threshold":0.001,"topk":50})
diag_cfg["trainer"].update({"save_all":True,"pretrain_model":None,"resume_model":False})
DIAG_CONFIG=MONODETR_REPO/"configs/monodetr_r0_epoch185_qualification.yaml"
DIAG_CONFIG.write_text(yaml.safe_dump(diag_cfg,sort_keys=False))
PREDICTION_DIR=RUN_DIR/"outputs/data"
shutil.rmtree(PREDICTION_DIR,ignore_errors=True)
run_logged([sys.executable,"-u","tools/train_val.py","--config",DIAG_CONFIG,"--evaluate_only"],MONODETR_REPO)
prediction_files=list(PREDICTION_DIR.glob("*.txt"))
if len(prediction_files) != 3769: raise RuntimeError(f"Expected 3769 prediction files, found {len(prediction_files)}")

QUALIFICATION_DIR=OUTPUT_ROOT/"r0_epoch185_locked_qualification"
run([sys.executable,"-u","scripts/audit_product_prediction_geometry.py","--dataset-root",MONODETR_KITTI,"--split-file",SPLIT_DIR/"val.txt","--prediction-dir",PREDICTION_DIR,"--output-dir",QUALIFICATION_DIR,"--checkpoint",SELECTED_CHECKPOINT,"--expected-checkpoint-sha256",SELECTED_SHA256,"--expected-images","3769","--score-threshold","0.001","--match-iou-threshold","0.5"],cwd=MOBILE_REPO)
summary=json.loads((QUALIFICATION_DIR/"nearby_geometry_summary.json").read_text())
print(json.dumps(summary["classes"],indent=2))
display(pd.read_csv(QUALIFICATION_DIR/"geometry_summary.csv"))

YAW_DIR=QUALIFICATION_DIR/"yaw_diagnostics"
run([sys.executable,"-u","scripts/evaluate_yaw_diagnostics.py","--matched-csv",QUALIFICATION_DIR/"matched_geometry.csv","--output-dir",YAW_DIR,"--split","val"],cwd=MOBILE_REPO)
display(pd.read_csv(YAW_DIR/"yaw_diagnostic_summary_val.csv"))

FALSE_NEGATIVE_DIR=QUALIFICATION_DIR/"pedestrian_false_negative_diagnostic"
run([sys.executable,"-u","scripts/diagnose_a2_pedestrian_false_negatives.py","--dataset-root",MONODETR_KITTI,"--split-file",SPLIT_DIR/"val.txt","--prediction-dir",PREDICTION_DIR,"--output-dir",FALSE_NEGATIVE_DIR,"--checkpoint",SELECTED_CHECKPOINT,"--expected-checkpoint-sha256",SELECTED_SHA256,"--expected-images","3769","--score-threshold","0.001","--iou-threshold","0.5","--weak-iou-threshold","0.1"],cwd=MOBILE_REPO)
fn_summary=json.loads((FALSE_NEGATIVE_DIR/"a2_pedestrian_false_negative_summary.json").read_text())
print(json.dumps(fn_summary,indent=2))
print("Return:",QUALIFICATION_DIR)
